# Chapter 17: LangGraph 워크플로우 테스팅 (Workflow Testing)

LangGraph 워크플로우를 **pytest**로 체계적으로 테스트하는 방법을 학습합니다.

**Sections:**
- 17.0 Setup & Environment
- 17.1 이메일 처리 그래프 (Rule-based email classifier)
- 17.2 Pytest 기초 (parametrize, %%writefile, !pytest)
- 17.3 노드 단위 테스트 (graph.nodes, update_state)
- 17.4 AI 노드 전환 (LLM + with_structured_output)
- 17.5 AI 테스트 전략 (Range-based assertions)
- 17.6 LLM-as-a-Judge (Golden examples, similarity scoring)
- Final Exercises

---
## 17.0 Setup & Environment

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL_NAME:", os.getenv("OPENAI_MODEL_NAME"))

OPENAI_API_KEY: a703ad29...
OPENAI_BASE_URL: https://lgcns-assetization3-japan-east.openai.azure.com/openai/v1
OPENAI_MODEL_NAME: gpt-5.1


In [2]:
# 패키지 확인
from importlib.metadata import version

print(f"langgraph: {version('langgraph')}")
print(f"langchain: {version('langchain')}")
print(f"pytest:    {version('pytest')}")

langgraph: 1.1.3
langchain: 1.2.13
pytest:    9.0.2


---
## 17.1 이메일 처리 그래프 — Rule-based Email Classifier

먼저 **규칙 기반(rule-based)** 이메일 분류기를 만든다.
모든 노드가 결정적(deterministic)이므로 테스트가 쉽다.

```
START → categorize_email → assign_priority → generate_response → END
```

- `categorize_email` — 키워드로 카테고리 분류 (urgent / spam / normal)
- `assign_priority` — 카테고리 기반 우선순위 (high / low / medium)
- `generate_response` — 카테고리별 정형 응답 생성

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# --- State 정의 ---
class EmailState(TypedDict):
    email: str
    category: str
    priority: str
    response: str


# --- 노드 함수 (모두 규칙 기반, LLM 없음) ---
def categorize_email(state: EmailState) -> dict:
    """키워드 기반 이메일 카테고리 분류"""
    email = state["email"].lower()
    if "urgent" in email or "asap" in email:
        return {"category": "urgent"}
    elif "offer" in email or "discount" in email:
        return {"category": "spam"}
    return {"category": "normal"}


def assign_priority(state: EmailState) -> dict:
    """카테고리 기반 우선순위 할당"""
    category = state["category"]
    if category == "urgent":
        return {"priority": "high"}
    elif category == "spam":
        return {"priority": "low"}
    return {"priority": "medium"}


def generate_response(state: EmailState) -> dict:
    """카테고리별 정형 응답 생성"""
    templates = {
        "urgent": "We received your urgent request and will respond within 1 hour.",
        "spam": "This email has been classified as spam.",
        "normal": "Thank you for your email. We will respond within 24 hours.",
    }
    return {"response": templates.get(state["category"], "Email received.")}


# --- 그래프 구성 ---
def build_email_graph():
    builder = StateGraph(EmailState)

    builder.add_node("categorize_email", categorize_email)
    builder.add_node("assign_priority", assign_priority)
    builder.add_node("generate_response", generate_response)

    builder.add_edge(START, "categorize_email")
    builder.add_edge("categorize_email", "assign_priority")
    builder.add_edge("assign_priority", "generate_response")
    builder.add_edge("generate_response", END)

    return builder.compile()


graph = build_email_graph()

In [4]:
# 전체 그래프 실행 테스트
result = graph.invoke({"email": "URGENT: Server is down, fix ASAP!"})
print(f"Category: {result['category']}")
print(f"Priority: {result['priority']}")
print(f"Response: {result['response']}")

Category: urgent
Priority: high
Response: We received your urgent request and will respond within 1 hour.


In [5]:
# 스팸 이메일 테스트
result2 = graph.invoke({"email": "Special offer! 50% discount today only!"})
print(f"Category: {result2['category']}")
print(f"Priority: {result2['priority']}")
print(f"Response: {result2['response']}")

Category: spam
Priority: low
Response: This email has been classified as spam.


In [6]:
# 일반 이메일 테스트
result3 = graph.invoke({"email": "Hi, I have a question about my order."})
print(f"Category: {result3['category']}")
print(f"Priority: {result3['priority']}")
print(f"Response: {result3['response']}")

Category: normal
Priority: medium
Response: Thank you for your email. We will respond within 24 hours.


---
## 17.2 Pytest 기초 — parametrize + %%writefile

노트북 안에서 **pytest**를 실행하는 패턴:

1. `%%writefile main.py` — 테스트 대상 코드를 파일로 저장
2. `%%writefile tests.py` — 테스트 코드를 파일로 저장
3. `!pytest tests.py -v` — 테스트 실행

핵심 개념:
- `@pytest.mark.parametrize` — 여러 입력/기대값을 한 번에 테스트
- `assert` — 기대값과 실제값 비교

In [7]:
%%writefile main.py
"""이메일 처리 그래프 — 규칙 기반 (Rule-based)"""

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class EmailState(TypedDict):
    email: str
    category: str
    priority: str
    response: str


def categorize_email(state: EmailState) -> dict:
    email = state["email"].lower()
    if "urgent" in email or "asap" in email:
        return {"category": "urgent"}
    elif "offer" in email or "discount" in email:
        return {"category": "spam"}
    return {"category": "normal"}


def assign_priority(state: EmailState) -> dict:
    category = state["category"]
    if category == "urgent":
        return {"priority": "high"}
    elif category == "spam":
        return {"priority": "low"}
    return {"priority": "medium"}


def generate_response(state: EmailState) -> dict:
    templates = {
        "urgent": "We received your urgent request and will respond within 1 hour.",
        "spam": "This email has been classified as spam.",
        "normal": "Thank you for your email. We will respond within 24 hours.",
    }
    return {"response": templates.get(state["category"], "Email received.")}


def build_email_graph():
    builder = StateGraph(EmailState)

    builder.add_node("categorize_email", categorize_email)
    builder.add_node("assign_priority", assign_priority)
    builder.add_node("generate_response", generate_response)

    builder.add_edge(START, "categorize_email")
    builder.add_edge("categorize_email", "assign_priority")
    builder.add_edge("assign_priority", "generate_response")
    builder.add_edge("generate_response", END)

    return builder.compile()

Writing main.py


In [8]:
%%writefile tests.py
"""이메일 처리 그래프 테스트 — pytest + parametrize"""

import pytest
from main import build_email_graph, categorize_email, assign_priority, generate_response


# --- 전체 그래프 통합 테스트 (End-to-End) ---
@pytest.mark.parametrize(
    "email, expected_category, expected_priority",
    [
        ("URGENT: Server down!", "urgent", "high"),
        ("Fix this ASAP please", "urgent", "high"),
        ("Special offer! Buy now!", "spam", "low"),
        ("Get 50% discount today", "spam", "low"),
        ("Hi, I have a question", "normal", "medium"),
        ("Meeting tomorrow at 3pm", "normal", "medium"),
    ],
)
def test_email_pipeline(email, expected_category, expected_priority):
    graph = build_email_graph()
    result = graph.invoke({"email": email})

    assert result["category"] == expected_category
    assert result["priority"] == expected_priority
    assert len(result["response"]) > 0  # 응답이 비어있지 않음


# --- 개별 노드 단위 테스트 ---
def test_categorize_urgent():
    result = categorize_email({"email": "This is URGENT!", "category": "", "priority": "", "response": ""})
    assert result["category"] == "urgent"


def test_categorize_spam():
    result = categorize_email({"email": "Big discount sale!", "category": "", "priority": "", "response": ""})
    assert result["category"] == "spam"


def test_categorize_normal():
    result = categorize_email({"email": "Hello, how are you?", "category": "", "priority": "", "response": ""})
    assert result["category"] == "normal"


def test_priority_mapping():
    assert assign_priority({"email": "", "category": "urgent", "priority": "", "response": ""})["priority"] == "high"
    assert assign_priority({"email": "", "category": "spam", "priority": "", "response": ""})["priority"] == "low"
    assert assign_priority({"email": "", "category": "normal", "priority": "", "response": ""})["priority"] == "medium"


def test_response_templates():
    result = generate_response({"email": "", "category": "urgent", "priority": "", "response": ""})
    assert "1 hour" in result["response"]

    result = generate_response({"email": "", "category": "spam", "priority": "", "response": ""})
    assert "spam" in result["response"]

    result = generate_response({"email": "", "category": "normal", "priority": "", "response": ""})
    assert "24 hours" in result["response"]

Writing tests.py


In [9]:
!pytest tests.py -v

============================= test session starts ==============================
platform darwin -- Python 3.13.2, pytest-9.0.2, pluggy-1.6.0 -- /Users/ryu/Documents/ati-gdc/docs/online-docs/agent-master-class/labs/venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ryu/Documents/ati-gdc/docs/online-docs/agent-master-class/labs/chapter_17
plugins: langsmith-0.7.22, anyio-4.13.0
collecting ... 

collected 11 items                                                             

tests.py::test_email_pipeline[URGENT: Server down!-urgent-high] PASSED   [  9%]
tests.py::test_email_pipeline[Fix this ASAP please-urgent-high] PASSED   [ 18%]
tests.py::test_email_pipeline[Special offer! Buy now!-spam-low] PASSED   [ 27%]
tests.py::test_email_pipeline[Get 50% discount today-spam-low] PASSED    [ 36%]
tests.py::test_email_pipeline[Hi, I have a question-normal-medium] PASSED [ 45%]
tests.py::test_email_pipeline[Meeting tomorrow at 3pm-normal-medium] PASSED [ 54%]
tests.py::test_categorize_urgent PASSED                                  [ 63%]
tests.py::test_categorize_spam PASSED                                    [ 72%]
tests.py::test_categorize_normal PASSED                                  [ 81%]
tests.py::test_priority_mapping PASSED                                   [ 90%]
tests.py::test_response_templates PASSED                                 [100%]

============================== 11 

### Exercise 17.2

1. `test_email_pipeline`에 edge case를 추가하라: 빈 문자열, 특수문자만 있는 이메일 등.
2. `@pytest.fixture`를 사용하여 `graph` 인스턴스를 재사용하도록 리팩터링하라.
3. `pytest.raises`를 사용하여 잘못된 입력에 대한 예외 테스트를 작성하라.

In [10]:
# Try it yourself


---
## 17.3 노드 단위 테스트 — graph.nodes + update_state

전체 그래프가 아닌 **특정 노드만** 테스트하는 방법:

1. `graph.nodes["name"].invoke(state)` — 개별 노드 직접 실행
2. `graph.update_state(config, values, as_node="name")` — 특정 노드 이후부터 부분 실행
3. `MemorySaver` 체크포인터로 상태 추적

In [11]:
from langgraph.checkpoint.memory import MemorySaver

# 체크포인터가 있는 그래프 빌드
def build_email_graph_with_memory():
    builder = StateGraph(EmailState)

    builder.add_node("categorize_email", categorize_email)
    builder.add_node("assign_priority", assign_priority)
    builder.add_node("generate_response", generate_response)

    builder.add_edge(START, "categorize_email")
    builder.add_edge("categorize_email", "assign_priority")
    builder.add_edge("assign_priority", "generate_response")
    builder.add_edge("generate_response", END)

    return builder.compile(checkpointer=MemorySaver())


graph_mem = build_email_graph_with_memory()

In [12]:
# 방법 1: graph.nodes["name"].invoke() — 개별 노드 직접 호출
# 노드 함수를 직접 실행하는 것과 동일

test_state = {"email": "URGENT: Need help ASAP", "category": "", "priority": "", "response": ""}

# categorize_email 노드만 실행
cat_result = graph.nodes["categorize_email"].invoke(test_state)
print(f"categorize_email result: {cat_result}")

# assign_priority 노드만 실행 (category가 필요)
prio_state = {"email": "", "category": "urgent", "priority": "", "response": ""}
prio_result = graph.nodes["assign_priority"].invoke(prio_state)
print(f"assign_priority result: {prio_result}")

# generate_response 노드만 실행
resp_state = {"email": "", "category": "spam", "priority": "low", "response": ""}
resp_result = graph.nodes["generate_response"].invoke(resp_state)
print(f"generate_response result: {resp_result}")

categorize_email result: {'category': 'urgent'}
assign_priority result: {'priority': 'high'}
generate_response result: {'response': 'This email has been classified as spam.'}


In [13]:
# 방법 2: update_state + as_node — 부분 실행
# 특정 노드가 실행된 것처럼 상태를 주입하고, 나머지만 실행

config = {"configurable": {"thread_id": "test_partial_1"}}

# 1단계: 초기 상태로 전체 실행
result = graph_mem.invoke(
    {"email": "Hello, normal email here"},
    config=config,
)
print(f"전체 실행 결과: category={result['category']}, priority={result['priority']}")
print(f"Response: {result['response']}")

전체 실행 결과: category=normal, priority=medium
Response: Thank you for your email. We will respond within 24 hours.


In [14]:
# 2단계: update_state로 categorize_email 결과를 강제 변경
# as_node="categorize_email" → 이 노드가 실행된 것처럼 상태 주입

config2 = {"configurable": {"thread_id": "test_partial_2"}}

# 먼저 초기 상태를 넣어야 update_state 가능
graph_mem.invoke({"email": "Hello, normal email"}, config=config2)

# categorize_email이 "urgent"를 반환한 것처럼 상태 강제 주입
graph_mem.update_state(
    config2,
    {"category": "urgent"},
    as_node="categorize_email",
)

# 현재 상태 확인
current = graph_mem.get_state(config2)
print(f"주입 후 상태: category={current.values['category']}")
print(f"다음 실행 노드: {current.next}")

주입 후 상태: category=urgent
다음 실행 노드: ('assign_priority',)


In [15]:
# 3단계: 나머지 노드만 실행 (assign_priority → generate_response)
result_partial = graph_mem.invoke(None, config=config2)
print(f"부분 실행 결과: category={result_partial['category']}, priority={result_partial['priority']}")
print(f"Response: {result_partial['response']}")

# category를 "urgent"로 강제했으므로 priority가 "high"가 됨
assert result_partial["priority"] == "high"
print("\nassert 통과! update_state로 부분 실행 성공")

부분 실행 결과: category=urgent, priority=high
Response: We received your urgent request and will respond within 1 hour.

assert 통과! update_state로 부분 실행 성공


### Exercise 17.3

1. `graph.nodes` 딕셔너리의 모든 키를 출력하고, 각 노드를 개별 실행해 보라.
2. `update_state(as_node="assign_priority")`로 우선순위를 강제 변경한 후 `generate_response`만 실행하라.
3. `get_state_history()`를 사용하여 부분 실행 전후의 상태 변화를 추적하라.

In [16]:
# Try it yourself


---
## 17.4 AI 노드 전환 — LLM + with_structured_output

규칙 기반 노드를 **AI 기반**으로 교체한다.

핵심:
- `with_structured_output(BaseModel)` — LLM 출력을 Pydantic 모델로 강제
- `Literal[...]` — 허용되는 값 제한
- 같은 StateGraph 구조, 노드 함수만 교체

In [17]:
import os
from typing import Literal
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# --- Pydantic 출력 스키마 ---
class CategoryOutput(BaseModel):
    """이메일 카테고리 분류 결과"""
    category: Literal["urgent", "spam", "normal"] = Field(
        description="The email category"
    )


class PriorityOutput(BaseModel):
    """우선순위 할당 결과"""
    priority: Literal["high", "medium", "low"] = Field(
        description="The priority level"
    )


class ResponseOutput(BaseModel):
    """이메일 응답 생성 결과"""
    response: str = Field(
        description="Professional response to the email"
    )


# --- Structured LLM ---
category_llm = llm.with_structured_output(CategoryOutput)
priority_llm = llm.with_structured_output(PriorityOutput)
response_llm = llm.with_structured_output(ResponseOutput)

print("Structured LLMs ready!")

Structured LLMs ready!


In [18]:
# --- AI 기반 노드 함수 ---
def ai_categorize_email(state: EmailState) -> dict:
    """LLM 기반 이메일 카테고리 분류"""
    result = category_llm.invoke(
        f"Classify this email into one of: urgent, spam, normal.\n\nEmail: {state['email']}"
    )
    return {"category": result.category}


def ai_assign_priority(state: EmailState) -> dict:
    """LLM 기반 우선순위 할당"""
    result = priority_llm.invoke(
        f"Assign priority (high/medium/low) for a '{state['category']}' email.\n\nEmail: {state['email']}"
    )
    return {"priority": result.priority}


def ai_generate_response(state: EmailState) -> dict:
    """LLM 기반 응답 생성"""
    result = response_llm.invoke(
        f"Write a professional response for this {state['category']} priority email.\n\nEmail: {state['email']}"
    )
    return {"response": result.response}


# --- AI 그래프 구성 ---
def build_ai_email_graph():
    builder = StateGraph(EmailState)

    builder.add_node("categorize_email", ai_categorize_email)
    builder.add_node("assign_priority", ai_assign_priority)
    builder.add_node("generate_response", ai_generate_response)

    builder.add_edge(START, "categorize_email")
    builder.add_edge("categorize_email", "assign_priority")
    builder.add_edge("assign_priority", "generate_response")
    builder.add_edge("generate_response", END)

    return builder.compile()


ai_graph = build_ai_email_graph()

In [19]:
# AI 그래프 실행
result = ai_graph.invoke({"email": "URGENT: Production database is corrupted, need immediate fix!"})
print(f"Category: {result['category']}")
print(f"Priority: {result['priority']}")
print(f"Response: {result['response'][:200]}")

Category: urgent
Priority: high
Response: Subject: Re: URGENT: Production database is corrupted, need immediate fix

Hi [Name],

Thank you for flagging this immediately.

I understand the urgency. Here is what I’m doing right now:

1. **Triag


In [20]:
# 스팸 이메일 AI 분류
result2 = ai_graph.invoke({"email": "Congratulations! You won $1,000,000! Click here to claim your prize!"})
print(f"Category: {result2['category']}")
print(f"Priority: {result2['priority']}")
print(f"Response: {result2['response'][:200]}")

Category: spam
Priority: high
Response: Subject: Re: Notification of Prize

To whom it may concern,

I am writing in response to your recent message claiming that I have won $1,000,000 and must click a link to claim the prize.

Please be ad


### Exercise 17.4

1. `CategoryOutput`에 `"inquiry"` 카테고리를 추가하고, 질문 이메일이 올바르게 분류되는지 테스트하라.
2. `with_structured_output`의 `method` 파라미터를 `"json_mode"`로 변경해 보라.
3. 규칙 기반과 AI 기반 결과를 동일 이메일 10개로 비교하라. 일치율은?

In [21]:
# Try it yourself


---
## 17.5 AI 테스트 전략 — Range-based Assertions

AI 출력은 **비결정적(non-deterministic)** 이다.
정확히 일치하는 값 대신 **범위(range)** 로 검증한다:

- `assert result in ["urgent", "spam", "normal"]` — 유효값 범위
- `assert min_len <= len(response) <= max_len` — 길이 범위
- `assert score >= threshold` — 최소 품질 기준
- 여러 번 실행 후 **성공률** 검증

In [22]:
%%writefile main_ai.py
"""AI 기반 이메일 처리 그래프"""

import os
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END

load_dotenv(dotenv_path="../.env")

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class EmailState(TypedDict):
    email: str
    category: str
    priority: str
    response: str


class CategoryOutput(BaseModel):
    category: Literal["urgent", "spam", "normal"] = Field(description="The email category")


class PriorityOutput(BaseModel):
    priority: Literal["high", "medium", "low"] = Field(description="The priority level")


class ResponseOutput(BaseModel):
    response: str = Field(description="Professional response to the email")


category_llm = llm.with_structured_output(CategoryOutput)
priority_llm = llm.with_structured_output(PriorityOutput)
response_llm = llm.with_structured_output(ResponseOutput)


def ai_categorize_email(state: EmailState) -> dict:
    result = category_llm.invoke(
        f"Classify this email into one of: urgent, spam, normal.\n\nEmail: {state['email']}"
    )
    return {"category": result.category}


def ai_assign_priority(state: EmailState) -> dict:
    result = priority_llm.invoke(
        f"Assign priority (high/medium/low) for a '{state['category']}' email.\n\nEmail: {state['email']}"
    )
    return {"priority": result.priority}


def ai_generate_response(state: EmailState) -> dict:
    result = response_llm.invoke(
        f"Write a short professional response for this {state['category']} priority email.\n\nEmail: {state['email']}"
    )
    return {"response": result.response}


def build_ai_email_graph():
    builder = StateGraph(EmailState)

    builder.add_node("categorize_email", ai_categorize_email)
    builder.add_node("assign_priority", ai_assign_priority)
    builder.add_node("generate_response", ai_generate_response)

    builder.add_edge(START, "categorize_email")
    builder.add_edge("categorize_email", "assign_priority")
    builder.add_edge("assign_priority", "generate_response")
    builder.add_edge("generate_response", END)

    return builder.compile()

Writing main_ai.py


In [23]:
%%writefile tests_ai.py
"""AI 기반 이메일 처리 테스트 — Range-based Assertions"""

import pytest
from main_ai import build_ai_email_graph


VALID_CATEGORIES = {"urgent", "spam", "normal"}
VALID_PRIORITIES = {"high", "medium", "low"}


@pytest.fixture
def ai_graph():
    return build_ai_email_graph()


# --- 유효값 범위 테스트 ---
@pytest.mark.parametrize(
    "email",
    [
        "URGENT: Server is down, fix immediately!",
        "Congratulations! You won a free iPhone!",
        "Hi, could you send me the Q3 report?",
    ],
)
def test_output_in_valid_range(ai_graph, email):
    """AI 출력이 유효한 값 범위 안에 있는지 확인"""
    result = ai_graph.invoke({"email": email})

    assert result["category"] in VALID_CATEGORIES, f"Invalid category: {result['category']}"
    assert result["priority"] in VALID_PRIORITIES, f"Invalid priority: {result['priority']}"
    assert len(result["response"]) > 10, "Response too short"
    assert len(result["response"]) < 2000, "Response too long"


# --- 명확한 케이스: 기대값 일치 테스트 ---
@pytest.mark.parametrize(
    "email, expected_category",
    [
        ("CRITICAL EMERGENCY: Data breach detected, all systems compromised!", "urgent"),
        ("FREE VIAGRA! Buy now! Limited offer! Click here!", "spam"),
        ("Hi team, attaching the meeting notes from today.", "normal"),
    ],
)
def test_clear_cases_match_expected(ai_graph, email, expected_category):
    """명확한 케이스는 AI도 정확히 맞춰야 한다"""
    result = ai_graph.invoke({"email": email})
    assert result["category"] == expected_category


# --- 응답 길이 범위 테스트 ---
def test_response_length_range(ai_graph):
    """응답 길이가 적절한 범위 안에 있는지 확인"""
    result = ai_graph.invoke({"email": "Please help me reset my password."})

    min_length = 20
    max_length = 1000
    actual_length = len(result["response"])

    assert min_length <= actual_length <= max_length, (
        f"Response length {actual_length} out of range [{min_length}, {max_length}]"
    )


# --- 일관성 테스트: 동일 입력 N회 실행 ---
def test_consistency_over_runs(ai_graph):
    """동일 입력을 3번 실행하여 카테고리 일관성 확인"""
    email = "URGENT: Production is completely down!"
    categories = []

    for _ in range(3):
        result = ai_graph.invoke({"email": email})
        categories.append(result["category"])

    # 3번 중 최소 2번은 동일해야 함 (66% 일관성)
    from collections import Counter
    most_common_count = Counter(categories).most_common(1)[0][1]
    assert most_common_count >= 2, f"Inconsistent results: {categories}"

Writing tests_ai.py


In [24]:
!pytest tests_ai.py -v

============================= test session starts ==============================
platform darwin -- Python 3.13.2, pytest-9.0.2, pluggy-1.6.0 -- /Users/ryu/Documents/ati-gdc/docs/online-docs/agent-master-class/labs/venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ryu/Documents/ati-gdc/docs/online-docs/agent-master-class/labs/chapter_17
plugins: langsmith-0.7.22, anyio-4.13.0
collecting ... 

collected 8 items                                                              

tests_ai.py::test_output_in_valid_range[URGENT: Server is down, fix immediately!] 

PASSED [ 12%]
tests_ai.py::test_output_in_valid_range[Congratulations! You won a free iPhone!] 

PASSED [ 25%]
tests_ai.py::test_output_in_valid_range[Hi, could you send me the Q3 report?] 

PASSED [ 37%]
tests_ai.py::test_clear_cases_match_expected[CRITICAL EMERGENCY: Data breach detected, all systems compromised!-urgent] 

...

PASSED [ 62%]
tests_ai.py::test_clear_cases_match_expected[Hi team, attaching the meeting notes from today.-normal] 

PASSED [ 75%]
tests_ai.py::test_response_length_range 

PASSED                           [ 87%]
tests_ai.py::test_consistency_over_runs 

PASSED                           [100%]

=================================== FAILURES ===================================
_ test_clear_cases_match_expected[CRITICAL EMERGENCY: Data breach detected, all systems compromised!-urgent] _

ai_graph = <langgraph.graph.state.CompiledStateGraph object at 0x10c14ab10>
email = 'CRITICAL EMERGENCY: Data breach detected, all systems compromised!'
expected_category = 'urgent'



    @pytest.mark.parametrize(
        "email, expected_category",
        [
            ("CRITICAL EMERGENCY: Data breach detected, all systems compromised!", "urgent"),
            ("FREE VIAGRA! Buy now! Limited offer! Click here!", "spam"),
            ("Hi team, attaching the meeting notes from today.", "normal"),
        ],
    )
    def test_clear_cases_match_expected(ai_graph, email, expected_category):
        """명확한 케이스는 AI도 정확히 맞춰야 한다"""
>       result = ai_graph.invoke({"email": email})
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

tests_ai.py:46: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
../venv/lib/python3.13/site-packages/langgraph/pregel/main.py:3292: in invoke
    for chunk in self.stream(
../venv/lib/python3.13/site-packages/langgraph/pregel/main.py:2725: in stream
    for _ in runner.tick(
../venv/lib/python3.13/site-packages/langgraph/pregel/_runner.py:167: in tick
    run_with_retry(
../venv/lib/python3.13/site-packages/

### Exercise 17.5

1. `test_consistency_over_runs`의 실행 횟수를 10으로 늘리고, 80% 이상 일관성을 요구하라.
2. 모호한 이메일("Could you please handle this quickly?")로 테스트하여 AI 분류의 한계를 탐구하라.
3. `temperature=0`을 설정하여 일관성이 개선되는지 비교하라.

In [25]:
# Try it yourself


---
## 17.6 LLM-as-a-Judge — Golden Examples + Similarity Scoring

AI 생성 응답의 **품질**을 평가하려면 사람 대신 **LLM이 판정자** 역할을 한다.

패턴:
1. **Golden Examples** — 이상적인 응답 예시 (RESPONSE_EXAMPLES dict)
2. **Judge LLM** — 생성 응답을 golden example과 비교하여 유사도 점수 부여
3. **Threshold** — 점수가 기준(70) 이상이면 통과

```
SimilarityScoreOutput:
  score: int (gt=0, lt=100)
  reasoning: str
```

In [26]:
import os
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# --- Golden Examples: 카테고리별 이상적 응답 ---
RESPONSE_EXAMPLES = {
    "urgent": (
        "Thank you for alerting us. We have escalated this to our on-call team "
        "and will provide an update within 1 hour. Please do not hesitate to "
        "call our emergency hotline if the situation worsens."
    ),
    "spam": (
        "This message has been identified as unsolicited commercial email. "
        "No action is required on your part. It has been moved to spam."
    ),
    "normal": (
        "Thank you for your email. We have received your message and a team "
        "member will respond within 24 hours during business days. If your "
        "matter is urgent, please call our support line."
    ),
}


# --- Similarity Score 출력 스키마 ---
class SimilarityScoreOutput(BaseModel):
    """LLM Judge가 반환하는 유사도 점수"""
    score: int = Field(gt=0, lt=100, description="Similarity score between 1 and 99")
    reasoning: str = Field(description="Brief explanation of the score")


judge_llm = llm.with_structured_output(SimilarityScoreOutput)


# --- Judge 함수 ---
def judge_response(generated: str, golden: str) -> SimilarityScoreOutput:
    """LLM이 생성된 응답과 골든 응답의 유사도를 평가"""
    prompt = f"""You are an expert quality evaluator. Compare the generated response 
with the golden (ideal) response and score their similarity.

Consider:
- Tone and professionalism (30%)
- Key information coverage (40%)
- Appropriate length and format (30%)

Golden Response:
{golden}

Generated Response:
{generated}

Score from 1 to 99 where:
- 80-99: Excellent match
- 60-79: Good match with minor differences
- 40-59: Acceptable but missing key elements
- 1-39: Poor match"""

    return judge_llm.invoke(prompt)


print("Judge function ready!")

Judge function ready!


In [27]:
# Judge 테스트: urgent 이메일 응답 평가
ai_result = ai_graph.invoke({"email": "CRITICAL: All servers are down! Need immediate help!"})

print(f"Category: {ai_result['category']}")
print(f"AI Response: {ai_result['response'][:200]}")
print()

# Golden example과 비교
golden = RESPONSE_EXAMPLES.get(ai_result["category"], RESPONSE_EXAMPLES["normal"])
score_result = judge_response(ai_result["response"], golden)

print(f"Similarity Score: {score_result.score}")
print(f"Reasoning: {score_result.reasoning}")

Category: urgent
AI Response: Subject: Re: CRITICAL: All servers are down! Need immediate help!

Thank you for alerting us.

We’re treating this as our top priority. Here’s what we’re doing right now:

1. **Immediate Actions**  
-



Similarity Score: 62
Reasoning: Tone/professionalism: Both are professional, polite, and appropriately urgent. The generated response is more formal (email-style with subject line and sign-off) and detailed but still aligned with the emergency tone. This is a good match, though more elaborate than the concise golden response. (≈25/30)

Key information coverage: Both acknowledge the alert and indicate escalation/engagement of on-call or equivalent teams. Both commit to providing an update within a defined timeframe; the generated response promises an update within 15 minutes (stricter than the golden response’s 1 hour). However, the golden explicitly mentions calling an emergency hotline if the situation worsens, which is missing from the generated response. The generated response adds extra useful operational detail and information requested from the user, which is not required but not conflicting. Missing the hotline/escalation contact is the main gap. (≈24/40)

Length/format: The gol

In [28]:
# 모든 카테고리에 대해 Judge 실행
test_emails = {
    "urgent": "EMERGENCY: Payment system is completely down, customers cannot pay!",
    "spam": "You have been selected for a $500 gift card! Claim now!",
    "normal": "Hello, I would like to inquire about your return policy.",
}

THRESHOLD = 70

for expected_cat, email in test_emails.items():
    result = ai_graph.invoke({"email": email})
    golden = RESPONSE_EXAMPLES[expected_cat]
    score_result = judge_response(result["response"], golden)

    status = "PASS" if score_result.score >= THRESHOLD else "FAIL"
    print(f"[{status}] {expected_cat}: score={score_result.score}, reason={score_result.reasoning[:80]}")

[PASS] urgent: score=72, reason=Tone and professionalism are quite aligned: both are formal, calm, and appropria


[FAIL] spam: score=20, reason=Tone is professional in both, but purpose is different: the golden response is a


[FAIL] normal: score=10, reason=Tone is polite and professional in both, but the content and purpose are entirel


In [29]:
%%writefile tests_judge.py
"""LLM-as-a-Judge 테스트 — Golden Examples + Similarity Scoring"""

import os
import pytest
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from main_ai import build_ai_email_graph

load_dotenv(dotenv_path="../.env")

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


RESPONSE_EXAMPLES = {
    "urgent": (
        "Thank you for alerting us. We have escalated this to our on-call team "
        "and will provide an update within 1 hour. Please do not hesitate to "
        "call our emergency hotline if the situation worsens."
    ),
    "spam": (
        "This message has been identified as unsolicited commercial email. "
        "No action is required on your part. It has been moved to spam."
    ),
    "normal": (
        "Thank you for your email. We have received your message and a team "
        "member will respond within 24 hours during business days. If your "
        "matter is urgent, please call our support line."
    ),
}


class SimilarityScoreOutput(BaseModel):
    score: int = Field(gt=0, lt=100, description="Similarity score between 1 and 99")
    reasoning: str = Field(description="Brief explanation of the score")


judge_llm = llm.with_structured_output(SimilarityScoreOutput)


def judge_response(generated: str, golden: str) -> SimilarityScoreOutput:
    prompt = f"""You are an expert quality evaluator. Compare the generated response 
with the golden (ideal) response and score their similarity.

Consider:
- Tone and professionalism (30%)
- Key information coverage (40%)
- Appropriate length and format (30%)

Golden Response:
{golden}

Generated Response:
{generated}

Score from 1 to 99."""
    return judge_llm.invoke(prompt)


@pytest.fixture
def ai_graph():
    return build_ai_email_graph()


THRESHOLD = 70


@pytest.mark.parametrize(
    "email, expected_category",
    [
        ("EMERGENCY: Payment system failure, customers affected!", "urgent"),
        ("FREE GIFT! You won a luxury vacation! Claim now!", "spam"),
        ("Hi, could you send me the project timeline?", "normal"),
    ],
)
def test_response_quality_above_threshold(ai_graph, email, expected_category):
    """AI 응답 품질이 threshold(70) 이상인지 LLM Judge로 검증"""
    result = ai_graph.invoke({"email": email})

    golden = RESPONSE_EXAMPLES[expected_category]
    score_result = judge_response(result["response"], golden)

    assert score_result.score >= THRESHOLD, (
        f"Score {score_result.score} < {THRESHOLD}. "
        f"Reasoning: {score_result.reasoning}"
    )


def test_judge_perfect_match():
    """Golden example 자체를 평가하면 높은 점수가 나와야 한다"""
    golden = RESPONSE_EXAMPLES["urgent"]
    score_result = judge_response(golden, golden)

    assert score_result.score >= 90, (
        f"Self-comparison should score >= 90, got {score_result.score}"
    )


def test_judge_poor_match():
    """완전히 다른 응답은 낮은 점수가 나와야 한다"""
    golden = RESPONSE_EXAMPLES["urgent"]
    poor_response = "lol ok whatever"
    score_result = judge_response(poor_response, golden)

    assert score_result.score < 40, (
        f"Poor response should score < 40, got {score_result.score}"
    )

Writing tests_judge.py


In [30]:
!pytest tests_judge.py -v

============================= test session starts ==============================
platform darwin -- Python 3.13.2, pytest-9.0.2, pluggy-1.6.0 -- /Users/ryu/Documents/ati-gdc/docs/online-docs/agent-master-class/labs/venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ryu/Documents/ati-gdc/docs/online-docs/agent-master-class/labs/chapter_17
plugins: langsmith-0.7.22, anyio-4.13.0
collecting ... 

collected 5 items                                                              

tests_judge.py::test_response_quality_above_threshold[EMERGENCY: Payment system failure, customers affected!-urgent] 

PASSED [ 20%]
tests_judge.py::test_response_quality_above_threshold[FREE GIFT! You won a luxury vacation! Claim now!-spam] 

FAILED [ 40%]
tests_judge.py::test_response_quality_above_threshold[Hi, could you send me the project timeline?-normal] 

FAILED [ 60%]
tests_judge.py::test_judge_perfect_match 

PASSED                          [ 80%]
tests_judge.py::test_judge_poor_match 

PASSED                             [100%]

=================================== FAILURES ===================================
_ test_response_quality_above_threshold[FREE GIFT! You won a luxury vacation! Claim now!-spam] _

ai_graph = <langgraph.graph.state.CompiledStateGraph object at 0x10aa93ed0>
email = 'FREE GIFT! You won a luxury vacation! Claim now!'
expected_category = 'spam'

    @pytest.mark.parametrize(
        "email, expected_category",
        [
            ("EMERGENCY: Payment system failure, customers affected!", "urgent"),
            ("FREE GIFT! You won a luxury vacation! Claim now!", "spam"),
            ("Hi, could you send me the project timeline?", "normal"),
        ],
    )
    def test_response_quality_above_threshold(ai_graph, email, expected_category):
        """AI 응답 품질이 threshold(70) 이상인지 LLM Judge로 검증"""
        result = ai_graph.invoke({"email": email})
    
        golden = RESPONSE_EXAMPLES[expected_category]
        score_result = judge_response(result[

### Exercise 17.6

1. `RESPONSE_EXAMPLES`에 새로운 카테고리(`inquiry`)를 추가하고 Judge 테스트를 확장하라.
2. `THRESHOLD`를 80으로 올리면 몇 개의 테스트가 실패하는가? 프롬프트를 개선하여 통과시켜라.
3. `judge_response` 프롬프트에 구체적인 평가 기준(예: "must mention response time")을 추가하라.

In [31]:
# Try it yourself


---
## Final Exercises — 종합 실습

### 과제 1: 테스트 커버리지 확장 (★★☆)

규칙 기반 그래프의 `tests.py`에 다음 테스트를 추가하라:
- Edge case: 빈 문자열, 매우 긴 이메일(1000자 이상)
- 대소문자 혼합: "uRgEnT", "DISCOUNT"
- `@pytest.fixture`로 그래프 인스턴스 공유

In [32]:
# 과제 1: 여기에 작성


### 과제 2: AI 분류 정확도 벤치마크 (★★★)

이메일 20개(카테고리당 ~7개)를 준비하고:
1. 규칙 기반과 AI 기반 분류 결과를 비교
2. AI 정확도(기대값 대비)를 계산
3. 정확도 80% 이상이면 PASS로 판정

In [33]:
# 과제 2: 여기에 작성


### 과제 3: 커스텀 Judge 체인 (★★★)

다음 기준으로 응답을 평가하는 multi-criteria Judge를 만들어라:
1. Professionalism (0-100)
2. Completeness (0-100)
3. Tone appropriateness (0-100)
4. 가중 평균으로 최종 점수 산출

In [34]:
# 과제 3: 여기에 작성


### 과제 4: 회귀 테스트 스위트 (★★★)

`%%writefile tests_regression.py`로 다음을 포함하는 회귀 테스트를 작성하라:
1. 규칙 기반 전체 파이프라인 (6개 이상 케이스)
2. AI 유효값 범위 검증 (3개 이상 케이스)
3. LLM-as-a-Judge 품질 검증 (3개 이상 케이스)
4. `!pytest tests_regression.py -v --tb=short`로 실행

In [35]:
# 과제 4: 여기에 작성
